# v11 — perbandingan semua model tim

Membaca `ringkasan.json` semua setelan yang sudah dievaluasi (selain QUICK) di `cache/v11/<model>/<setelan>/`.
Hasil teman: ekstrak zip hasilnya (sel terakhir notebook model) ke `cache/v11/`; ringkasan sudah ada di dalam zip.
Pembanding v1 dan v3.1 dihitung dari cache yang sama (5 ulangan).

Urutan: (1) semua setelan, (2) antar model dengan resep yang sama (`bawaan`, berpasangan), (3) balancing vs bobot kelas
(arahan pembimbing), (4) tuning cross-fit per model, (5) kandidat: tanpa 04936 dan bootstrap bersarang, (6) simpan semua angka
ke `cache/v11/gabungan.json` (dibaca `laporan/build_report.py`). Dengan semua flag aktif, ±55 menit di CPU (terlama: tuning cross-fit BiGRU, 14 setelan).

In [1]:
REPO_URL = "https://github.com/farhanlhsn/WrapSihedaf.git"
BRANCH = "main"
DATA_URL = "https://github.com/farhanlhsn/WrapSihedaf/releases/download/v11-data/v11_data.zip"
HASIL_DI = None                # folder hasil bila bukan cache/v11 (mis. Google Drive), sama dengan notebook model
DATA_DI = None                 # isi hanya bila unduhan data gagal
JALANKAN_BANDING_BALANCING = True   # setiap setelan ber-balancing vs setelan "bawaan" model yang sama (berpasangan)
JALANKAN_TUNING = True         # pilih setelan terbaik secara cross-fit dari beberapa setelan SATU model
JALANKAN_TANPA_04936 = True    # protokol butir 10: hanya bila sebuah model diusulkan mengganti produksi
JALANKAN_BERSARANG = True      # protokol butir 10: bootstrap bersarang (butuh 5 ulangan lengkap)
MODEL_ACUAN = "bigru"          # model bawaan terbaik: pembanding berpasangan antar arsitektur (resep sama)
MODEL_TUNING = ["bigru", "tcn", "inceptiontime", "xgboost"]   # model dengan > 1 setelan tanpa balancing
KANDIDAT = ["bigru/wp100", "tcn/wp100"]                       # setelan yang diusulkan (protokol butir 10)

import json
import os
import subprocess
import sys
from pathlib import Path

if HASIL_DI and str(HASIL_DI).startswith("/content/drive") and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
if HASIL_DI:
    os.environ["SIHEDAF_HASIL"] = str(HASIL_DI)
if DATA_DI:
    os.environ["SIHEDAF_DATA"] = str(DATA_DI)
if not Path("bersama").exists():                         # Kaggle/Colab: clone repo dulu
    root = Path("/kaggle/working/WrapSihedaf") if Path("/kaggle/working").exists() else Path.cwd() / "WrapSihedaf"
    if not root.exists():
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(root)], check=True)
    os.chdir(root / "v11_model_tim")
assert Path.cwd().name == "v11_model_tim", f"cwd harus folder v11_model_tim, sekarang {Path.cwd()}"
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
pd.set_option("display.max_colwidth", None)   # teks vonis tidak terpotong

from bersama import data, evaluasi, runner

data.siapkan_data(DATA_URL)
d = data.muat()
print("hasil dibaca dari:", data.lokasi_hasil())

hasil dibaca dari: /mnt/HDD/03_Akademik_dan_Tugas/WrapSihedaf/cache/v11


## 1. Semua setelan (+ bias kelas cross-fit)

In [2]:
semua = evaluasi.kumpulkan_ringkasan()
print(f"{len(semua)} setelan ditemukan")
fmt = lambda v: f"{v[0]:.3f} [{v[1]:.2f}–{v[2]:.2f}]"
fmt_banding = lambda v: f"Δ {v[0]:+.3f} [{v[1]:+.2f}…{v[2]:+.2f}], P = {v[3]:.2f} → {evaluasi.vonis(v[3])}"
baris = []
for r in semua:
    m, b31, b1 = r["metrik"]["bias"], r["banding_v31"]["Macro-F1"], r["banding_v1"]["Macro-F1"]
    baris.append({"model": r["model"], "setelan": r["nama_setelan"],
                  "balancing": r["setelan"].get("balancing") or "-", "ulangan": len(r["ulangan"]),
                  "Macro-F1": fmt(m["Macro-F1"]), "_urut": m["Macro-F1"][0],
                  "P vs v3.1": f"{b31[3]:.2f} ({evaluasi.vonis(b31[3])})",
                  "P vs v1": f"{b1[3]:.2f} ({evaluasi.vonis(b1[3])})",
                  "AFL F1": fmt(m["AFL F1"]), "AFL R": fmt(m["AFL R"]), "Normal R": fmt(m["Normal R"]),
                  "ambang takikardia (bpm)": r["takikardia"]["model"]["ambang"]["tanpa variasi"],
                  "bias di batas": f"{r['bias']['di_batas']:.0%}", "detik/fold": round(r["waktu_per_fold"])})
pred_v31, _ = evaluasi.keputusan(d.oof_v31, d.folds, d.y)
acuan = evaluasi.tabel_metrik(d, {"v1": d.oof_v1.argmax(-1), "v31": pred_v31})
for k, nama in (("v31", "v3.1 (referensi, + bias)"), ("v1", "v1 (argmax)")):
    baris.append({"model": nama, "setelan": "-", "balancing": "-", "ulangan": 5, "Macro-F1": fmt(acuan[k]["Macro-F1"]),
                  "_urut": acuan[k]["Macro-F1"][0], "AFL F1": fmt(acuan[k]["AFL F1"]), "AFL R": fmt(acuan[k]["AFL R"]),
                  "Normal R": fmt(acuan[k]["Normal R"]),
                  "ambang takikardia (bpm)": evaluasi.AMBANG_V31_PAKET if k == "v31" else None})
tabel = pd.DataFrame(baris).sort_values("_urut", ascending=False).drop(columns="_urut").set_index(["model", "setelan"])
display(tabel)

31 setelan ditemukan


balancing  ulangan           Macro-F1  \
model                    setelan                                              
bigru                    lr5e4                -        5  0.833 [0.68–0.91]   
                         w1_wd1e3             -        5  0.833 [0.68–0.93]   
                         w1_wd1e2             -        5  0.832 [0.68–0.93]   
v3.1 (referensi, + bias) -                    -        5  0.832 [0.67–0.92]   
tcn                      wp100                -        5  0.830 [0.67–0.91]   
bigru                    wp100                -        5  0.830 [0.69–0.93]   
                         w1_bs512             -        5  0.828 [0.67–0.92]   
                         w1_ep20              -        5  0.827 [0.69–0.93]   
                         h16                  -        5  0.824 [0.68–0.90]   
                         w1_lr5e4             -        5  0.824 [0.68–0.92]   
                         bawaan               -        5  0.821 [0.68–0.87]   
                         bawaan_ulang         -        5  0.821 [0.68–0.87]   
                         wp075                -        5  0.821 [0.68–0.89]   
                         h64                  -        5  0.813 [0.67–0.84]   
tcn                      k16                  -        5  0.812 [0.68–0.84]   
bigru                    ep50                 -        5  0.797 [0.67–0.83]   
xgboost                  under            under        5  0.794 [0.65–0.84]   
tcn                      wp025                -        5  0.785 [0.65–0.83]   
                         bawaan_ulang         -        5  0.783 [0.65–0.82]   
bigru                    w1_h16               -        5  0.782 [0.68–0.87]   
                         over              over        5  0.776 [0.65–0.82]   
svm                      bawaan               -        5  0.773 [0.64–0.92]   
tcn                      d02                  -        5  0.770 [0.63–0.82]   
inceptiontime            bawaan               -        5  0.767 [0.64–0.86]   
                         wp100                -        5  0.766 [0.64–0.92]   
tcn                      dil124               -        5  0.762 [0.63–0.81]   
                         bawaan               -        5  0.752 [0.63–0.80]   
xgboost                  bobot_penuh          -        5  0.744 [0.63–0.79]   
v1 (argmax)              -                    -        5  0.744 [0.64–0.81]   
xgboost                  smote            smote        5  0.743 [0.62–0.79]   
                         bawaan               -        5  0.730 [0.62–0.78]   
minirocket               bawaan               -        5  0.709 [0.62–0.76]   
bigru                    under            under        5  0.599 [0.49–0.66]   

                                                     P vs v3.1  \
model                    setelan                                 
bigru                    lr5e4         0.61 (tidak terbedakan)   
                         w1_wd1e3      0.63 (tidak terbedakan)   
                         w1_wd1e2      0.57 (tidak terbedakan)   
v3.1 (referensi, + bias) -                                 NaN   
tcn                      wp100         0.38 (tidak terbedakan)   
bigru                    wp100         0.47 (tidak terbedakan)   
                         w1_bs512      0.32 (tidak terbedakan)   
                         w1_ep20       0.40 (tidak terbedakan)   
                         h16           0.20 (tidak terbedakan)   
                         w1_lr5e4      0.07 (tidak terbedakan)   
                         bawaan        0.34 (tidak terbedakan)   
                         bawaan_ulang  0.34 (tidak terbedakan)   
                         wp075         0.12 (tidak terbedakan)   
                         h64           0.28 (tidak terbedakan)   
tcn                      k16           0.30 (tidak terbedakan)   
bigru                    ep50          0.15 (tidak terbedakan)   
xgboost                  under              0.00 (lebih buruk)   
tcn               

In [3]:
# Ringkas: resep yang sama untuk semua model (bawaan) + kandidat. Alarm = % window Normal asli ≥ 120 bpm yang ditebak AFIB/AFL.
ringkas = {(r["model"], r["nama_setelan"]): r for r in semua
           if r["nama_setelan"] == "bawaan" or f"{r['model']}/{r['nama_setelan']}" in KANDIDAT}
display(pd.DataFrame({f"{m}/{s}": {
    "Macro-F1": fmt(r["metrik"]["bias"]["Macro-F1"]), "AFL F1": fmt(r["metrik"]["bias"]["AFL F1"]),
    "vs v3.1": fmt_banding(r["banding_v31"]["Macro-F1"]), "vs v1": fmt_banding(r["banding_v1"]["Macro-F1"]),
    "recall 08378": round(r["afl_per_pasien"]["bias"]["08378"], 2),
    "ambang (bpm)": " / ".join(map(str, r["takikardia"]["model"]["ambang"].values())),
    "alarm Normal ≥ 120": f"{1 - r['pita_detak']['Normal | 120–∞ bpm']['model'][0]:.1%}"}
    for (m, s), r in sorted(ringkas.items(), key=lambda x: -x[1]["metrik"]["bias"]["Macro-F1"][0])}).T)

,Macro-F1,AFL F1,vs v3.1,vs v1,recall 08378,ambang (bpm),alarm Normal ≥ 120
tcn/wp100,0.830 [0.67–0.91],0.623 [0.14–0.86],"Δ -0.002 [-0.02…+0.01], P = 0.38 → tidak terbedakan","Δ +0.086 [+0.02…+0.16], P = 0.99 → lebih baik",0.92,125 / 125 / 125 / 120,84.4%
bigru/wp100,0.830 [0.69–0.93],0.625 [0.21–0.92],"Δ -0.002 [-0.01…+0.02], P = 0.47 → tidak terbedakan","Δ +0.086 [+0.03…+0.17], P = 0.99 → lebih baik",1.0,120 / 120 / 120 / 115,90.2%
bigru/bawaan,0.821 [0.68–0.87],0.592 [0.16–0.73],"Δ -0.010 [-0.06…+0.02], P = 0.34 → tidak terbedakan","Δ +0.078 [+0.02…+0.13], P = 0.99 → lebih baik",0.74,120 / 120 / 120 / 120,86.6%
svm/bawaan,0.773 [0.64–0.92],0.487 [0.09–0.90],"Δ -0.059 [-0.12…+0.00], P = 0.07 → tidak terbedakan","Δ +0.029 [-0.01…+0.14], P = 0.91 → tidak terbedakan",0.98,140 / 125 / 125 / 120,78.4%
inceptiontime/bawaan,0.767 [0.64–0.86],0.446 [0.06–0.72],"Δ -0.065 [-0.12…-0.02], P = 0.01 → lebih buruk","Δ +0.023 [-0.01…+0.09], P = 0.86 → tidak terbedakan",0.79,120 / 130 / 130 / 125,72.2%
tcn/bawaan,0.752 [0.63–0.80],0.391 [0.02–0.53],"Δ -0.080 [-0.27…+0.00], P = 0.03 → lebih buruk","Δ +0.008 [-0.16…+0.09], P = 0.53 → tidak terbedakan",0.01,120 / 125 / 130 / 130,71.8%
xgboost/bawaan,0.730 [0.62–0.78],0.363 [0.04–0.51],"Δ -0.101 [-0.26…-0.02], P = 0.00 → lebih buruk","Δ -0.013 [-0.16…+0.05], P = 0.36 → tidak terbedakan",0.05,125 / 120 / 115 / 115,85.0%
minirocket/bawaan,0.709 [0.62–0.76],0.306 [0.04–0.46],"Δ -0.122 [-0.25…-0.03], P = 0.00 → lebih buruk","Δ -0.034 [-0.15…+0.03], P = 0.21 → tidak terbedakan",0.11,130 / 135 / 130 / 120,92.4%


## 2. Antar model dengan resep yang sama (`bawaan`, berpasangan)

Semua model `bawaan` memakai data, fold, seed, bobot kelas lunak, dan bias cross-fit yang sama, jadi selisihnya berasal dari
arsitektur. Pembanding: model bawaan terbaik (`MODEL_ACUAN`), dengan sampel bootstrap pasien yang sama.

In [4]:
pred_bawaan = {}
for r in semua:
    if r["nama_setelan"] == "bawaan":
        h = runner.muat_hasil(data.lokasi_hasil() / r["model"] / "bawaan", d)
        assert h.ulangan == list(range(data.N_ULANGAN)), r["model"]
        pred_bawaan[r["model"]] = evaluasi.keputusan(h.oof, d.folds, d.y)[0]
antar_model = {m: evaluasi.banding(d, p, pred_bawaan[MODEL_ACUAN]) for m, p in pred_bawaan.items() if m != MODEL_ACUAN}
display(pd.DataFrame({m: {k: fmt_banding(v) for k, v in b.items()} for m, b in antar_model.items()}).T
        .rename_axis(f"bawaan vs {MODEL_ACUAN} bawaan"))

,Macro-F1,AFL F1,AFL P,AFL R,Normal R
bawaan vs bigru bawaan,,,,,
inceptiontime,"Δ -0.055 [-0.08…-0.00], P = 0.02 → lebih buruk","Δ -0.146 [-0.23…+0.01], P = 0.03 → lebih buruk","Δ -0.211 [-0.25…+0.07], P = 0.24 → tidak terbedakan","Δ -0.077 [-0.19…+0.03], P = 0.12 → tidak terbedakan","Δ -0.003 [-0.02…+0.01], P = 0.39 → tidak terbedakan"
minirocket,"Δ -0.112 [-0.20…-0.04], P = 0.00 → lebih buruk","Δ -0.287 [-0.55…-0.09], P = 0.00 → lebih buruk","Δ -0.321 [-0.56…-0.04], P = 0.00 → lebih buruk","Δ -0.248 [-0.56…-0.03], P = 0.00 → lebih buruk","Δ -0.045 [-0.07…-0.03], P = 0.00 → lebih buruk"
svm,"Δ -0.048 [-0.08…+0.05], P = 0.16 → tidak terbedakan","Δ -0.106 [-0.20…+0.17], P = 0.25 → tidak terbedakan","Δ -0.211 [-0.24…+0.09], P = 0.20 → tidak terbedakan","Δ +0.057 [-0.04…+0.21], P = 0.66 → tidak terbedakan","Δ -0.021 [-0.05…-0.00], P = 0.01 → lebih buruk"
tcn,"Δ -0.070 [-0.21…-0.01], P = 0.00 → lebih buruk","Δ -0.201 [-0.63…-0.03], P = 0.00 → lebih buruk","Δ -0.179 [-0.62…-0.02], P = 0.00 → lebih buruk","Δ -0.204 [-0.65…-0.01], P = 0.00 → lebih buruk","Δ +0.009 [+0.00…+0.02], P = 1.00 → lebih baik"
xgboost,"Δ -0.091 [-0.21…-0.03], P = 0.00 → lebih buruk","Δ -0.229 [-0.59…-0.04], P = 0.00 → lebih buruk","Δ -0.257 [-0.58…-0.04], P = 0.00 → lebih buruk","Δ -0.202 [-0.62…+0.01], P = 0.04 → lebih buruk","Δ -0.019 [-0.05…-0.00], P = 0.00 → lebih buruk"


## 3. Balancing vs bobot kelas (arahan pembimbing)

In [5]:
hasil_balancing = {}
if JALANKAN_BANDING_BALANCING:
    # Jawaban langsung untuk "apakah balancing membantu?": model yang sama, fold & bootstrap pasien yang sama.
    for r in semua:
        if not r["setelan"].get("balancing"):
            continue
        ref = data.lokasi_hasil() / r["model"] / "bawaan"
        if not (ref / "setelan.json").exists():
            print(f"{r['model']}/{r['nama_setelan']}: setelan 'bawaan' belum ada, dilewati")
            continue
        if json.loads((ref / "setelan.json").read_text(encoding="utf-8"))["setelan"].get("balancing"):
            print(f"{r['model']}/bawaan ternyata memakai balancing, dilewati")
            continue
        b = evaluasi.banding_setelan(d, data.lokasi_hasil() / r["model"] / r["nama_setelan"], ref)
        hasil_balancing[f"{r['model']}/{r['nama_setelan']}"] = b
        print(f"\n{b['model']}: {b['kandidat']} vs {b['referensi']} (ulangan {b['ulangan']})")
        display(pd.DataFrame({m: [fmt_banding(v)] for m, v in b["banding"].items()}, index=["vs bawaan"]).T)
        display(pd.DataFrame(b["afl_per_pasien"]).round(2).rename_axis("recall AFL per pasien"))


bigru: over vs bawaan (ulangan [0, 1, 2, 3, 4])


,vs bawaan
Macro-F1,"Δ -0.045 [-0.07…-0.02], P = 0.00 → lebih buruk"
AFL F1,"Δ -0.128 [-0.22…-0.06], P = 0.00 → lebih buruk"
AFL P,"Δ -0.146 [-0.23…-0.06], P = 0.00 → lebih buruk"
AFL R,"Δ -0.103 [-0.19…-0.04], P = 0.00 → lebih buruk"
Normal R,"Δ -0.006 [-0.01…-0.00], P = 0.00 → lebih buruk"


,kandidat,referensi
recall AFL per pasien,,
04043,0.40,0.80
04908,0.81,0.98
04936,0.44,0.52
06426,0.15,0.29
07910,0.00,0.00
08215,0.00,0.00
08378,0.55,0.74



bigru: under vs bawaan (ulangan [0, 1, 2, 3, 4])


,vs bawaan
Macro-F1,"Δ -0.223 [-0.33…-0.14], P = 0.00 → lebih buruk"
AFL F1,"Δ -0.291 [-0.64…-0.04], P = 0.00 → lebih buruk"
AFL P,"Δ -0.412 [-0.77…-0.11], P = 0.00 → lebih buruk"
AFL R,"Δ -0.132 [-0.64…+0.10], P = 0.26 → tidak terbedakan"
Normal R,"Δ -0.184 [-0.28…-0.10], P = 0.00 → lebih buruk"


,kandidat,referensi
recall AFL per pasien,,
04043,0.00,0.80
04908,0.99,0.98
04936,0.62,0.52
06426,0.16,0.29
07910,0.00,0.00
08215,0.00,0.00
08378,0.03,0.74


... (2 outputs left out)

,kandidat,referensi
recall AFL per pasien,,
04043,0.40,0.20
04908,0.99,1.00
04936,0.53,0.49
06426,0.02,0.08
07910,0.01,0.04
08215,0.00,0.00
08378,0.07,0.05



xgboost: under vs bawaan (ulangan [0, 1, 2, 3, 4])


,vs bawaan
Macro-F1,"Δ +0.063 [+0.00…+0.18], P = 0.98 → lebih baik"
AFL F1,"Δ +0.191 [+0.01…+0.54], P = 0.98 → lebih baik"
AFL P,"Δ +0.231 [+0.02…+0.58], P = 0.99 → lebih baik"
AFL R,"Δ +0.167 [-0.05…+0.55], P = 0.85 → tidak terbedakan"
Normal R,"Δ -0.028 [-0.04…-0.02], P = 0.00 → lebih buruk"


,kandidat,referensi
recall AFL per pasien,,
04043,0.20,0.20
04908,0.89,1.00
04936,0.50,0.49
06426,0.11,0.08
07910,0.00,0.04
08215,0.00,0.00
08378,0.67,0.05


## 4. Tuning cross-fit per model

Untuk tiap fold uji dipakai setelan terbaik di fold lain, jadi tuning tidak mengintip data uji. Ini perkiraan jujur untuk
"model yang di-tuning"; setelan terbaik di tabel 1 dipilih setelah melihat hasil, jadi sedikit terlalu optimistis.
Setelan ber-balancing tidak ikut (dibahas di bagian 3).

In [6]:
hasil_tuning = {}
if JALANKAN_TUNING:
    for model in MODEL_TUNING:
        folders = [data.lokasi_hasil() / r["model"] / r["nama_setelan"] for r in semua
                   if r["model"] == model and not r["setelan"].get("balancing")]
        if len(folders) < 2:
            print(f"{model}: hanya {len(folders)} setelan, dilewati")
            continue
        t = evaluasi.pilih_crossfit(d, folders)
        hasil_tuning[model] = {"kandidat": [f.name for f in folders], **t,
                               "terpilih": [list(map(str, x)) for x in t["terpilih"]]}
        print(f"\n{model}: {len(folders)} setelan; terpilih per (ulangan, fold): {hasil_tuning[model]['terpilih']}")
        display(pd.DataFrame({k: {m: fmt(v[m]) for m in ["Macro-F1", "AFL F1", "AFL R", "Normal R"]}
                              for k, v in t["metrik"].items()}).T)
        display(pd.DataFrame({m: [fmt_banding(v)] for m, v in t["banding_v31"].items()}, index=["pilihan vs v3.1"]).T)


bigru: 14 setelan; terpilih per (ulangan, fold): [['lr5e4', 'ep50', 'lr5e4', 'ep50', 'wp075'], ['h16', 'h16', 'h16', 'h16', 'h16'], ['h64', 'h64', 'h16', 'h64', 'h64'], ['lr5e4', 'lr5e4', 'lr5e4', 'lr5e4', 'wp100'], ['wp075', 'w1_bs512', 'wp075', 'wp075', 'wp075']]


,Macro-F1,AFL F1,AFL R,Normal R
pilihan,0.818 [0.67–0.88],0.574 [0.14–0.76],0.507 [0.14–0.76],0.944 [0.89–0.97]
v31,0.832 [0.67–0.92],0.647 [0.17–0.89],0.592 [0.13–0.91],0.924 [0.84–0.97]


,pilihan vs v3.1
Macro-F1,"Δ -0.013 [-0.04…+0.01], P = 0.08 → tidak terbedakan"
AFL F1,"Δ -0.073 [-0.15…-0.02], P = 0.02 → lebih buruk"
AFL P,"Δ -0.046 [-0.15…-0.00], P = 0.02 → lebih buruk"
AFL R,"Δ -0.085 [-0.16…-0.00], P = 0.02 → lebih buruk"
Normal R,"Δ +0.020 [+0.00…+0.05], P = 1.00 → lebih baik"



tcn: 7 setelan; terpilih per (ulangan, fold): [['k16', 'k16', 'd02', 'k16', 'bawaan_ulang'], ['wp100', 'bawaan_ulang', 'wp100', 'wp100', 'wp100'], ['wp025', 'wp025', 'wp100', 'wp025', 'wp025'], ['d02', 'd02', 'k16', 'k16', 'k16'], ['wp100', 'wp100', 'wp100', 'wp025', 'wp100']]


,Macro-F1,AFL F1,AFL R,Normal R
pilihan,0.782 [0.65–0.82],0.471 [0.11–0.58],0.403 [0.09–0.55],0.953 [0.91–0.98]
v31,0.832 [0.67–0.92],0.647 [0.17–0.89],0.592 [0.13–0.91],0.924 [0.84–0.97]


,pilihan vs v3.1
Macro-F1,"Δ -0.050 [-0.21…+0.02], P = 0.25 → tidak terbedakan"
AFL F1,"Δ -0.176 [-0.65…+0.00], P = 0.06 → tidak terbedakan"
AFL P,"Δ -0.141 [-0.64…-0.02], P = 0.01 → lebih buruk"
AFL R,"Δ -0.189 [-0.67…+0.02], P = 0.25 → tidak terbedakan"
Normal R,"Δ +0.029 [+0.01…+0.07], P = 1.00 → lebih baik"


... (2 outputs left out)

,pilihan vs v3.1
Macro-F1,"Δ -0.066 [-0.11…-0.02], P = 0.01 → lebih buruk"
AFL F1,"Δ -0.204 [-0.34…-0.08], P = 0.00 → lebih buruk"
AFL P,"Δ -0.264 [-0.37…+0.04], P = 0.12 → tidak terbedakan"
AFL R,"Δ -0.138 [-0.19…-0.05], P = 0.01 → lebih buruk"
Normal R,"Δ +0.024 [-0.00…+0.07], P = 0.93 → tidak terbedakan"



xgboost: 2 setelan; terpilih per (ulangan, fold): [['bobot_penuh', 'bobot_penuh', 'bobot_penuh', 'bobot_penuh', 'bobot_penuh'], ['bobot_penuh', 'bobot_penuh', 'bobot_penuh', 'bawaan', 'bawaan'], ['bawaan', 'bawaan', 'bobot_penuh', 'bawaan', 'bawaan'], ['bobot_penuh', 'bobot_penuh', 'bobot_penuh', 'bobot_penuh', 'bobot_penuh'], ['bawaan', 'bobot_penuh', 'bobot_penuh', 'bobot_penuh', 'bawaan']]


,Macro-F1,AFL F1,AFL R,Normal R
pilihan,0.745 [0.63–0.79],0.405 [0.08–0.53],0.364 [0.06–0.53],0.930 [0.86–0.97]
v31,0.832 [0.67–0.92],0.647 [0.17–0.89],0.592 [0.13–0.91],0.924 [0.84–0.97]


,pilihan vs v3.1
Macro-F1,"Δ -0.087 [-0.23…-0.01], P = 0.00 → lebih buruk"
AFL F1,"Δ -0.242 [-0.69…-0.02], P = 0.00 → lebih buruk"
AFL P,"Δ -0.253 [-0.62…-0.03], P = 0.01 → lebih buruk"
AFL R,"Δ -0.228 [-0.73…+0.02], P = 0.06 → tidak terbedakan"
Normal R,"Δ +0.006 [-0.00…+0.01], P = 0.97 → lebih baik"


## 5. Kandidat: tanpa pasien 04936 dan bootstrap bersarang (protokol butir 10)

In [7]:
hasil_kandidat = {"tanpa_04936": None, "bersarang": {}}
if JALANKAN_TANPA_04936 or JALANKAN_BERSARANG:
    hk = {k: runner.muat_hasil(data.lokasi_hasil() / k, d) for k in KANDIDAT}
    assert all(h.ulangan == list(range(data.N_ULANGAN)) for h in hk.values())
    if JALANKAN_TANPA_04936:
        s = evaluasi.tanpa_04936(d, {"v31": d.oof_v31, **{k: h.oof for k, h in hk.items()}}, list(range(data.N_ULANGAN)))
        hasil_kandidat["tanpa_04936"] = s
        display(pd.DataFrame({k: {m: fmt(v[m]) for m in ["Macro-F1", "AFL F1", "AFL R", "Normal R"]} for k, v in s["rows"].items()}).T
                .rename_axis(f"tanpa 04936 ({s['n_pasien']} pasien, {s['n_pasien_afl']} pasien AFL)"))
        for k in KANDIDAT:
            print(f"Macro-F1 v3.1 − {k}: {fmt_banding(s['pairs_v31_minus'][k])}  (lebih baik = v3.1 lebih baik)")
    if JALANKAN_BERSARANG:
        for k, h in hk.items():
            b = evaluasi.bootstrap_bersarang(d, h.oof)
            hasil_kandidat["bersarang"][k] = b
            kk, v1 = b["kandidat"], b["vs_v1"]
            print(f"Bootstrap bersarang {k}: {kk['titik']:.3f} [{kk['ci'][0]:.2f}–{kk['ci'][1]:.2f}] | vs v1 "
                  f"{fmt_banding([v1['delta'], *v1['ci'], v1['p']])}")

,Macro-F1,AFL F1,AFL R,Normal R
"tanpa 04936 (22 pasien, 6 pasien AFL)",,,,
v31,0.859 [0.62–0.92],0.725 [0.02–0.90],0.663 [0.02–0.89],0.928 [0.86–0.97]
bigru/wp100,0.852 [0.63–0.90],0.690 [0.03–0.83],0.618 [0.02–0.82],0.940 [0.88–0.97]
tcn/wp100,0.840 [0.63–0.90],0.648 [0.02–0.83],0.600 [0.02–0.80],0.956 [0.92–0.98]


Macro-F1 v3.1 − bigru/wp100: Δ +0.007 [-0.04…+0.03], P = 0.60 → tidak terbedakan  (lebih baik = v3.1 lebih baik)
Macro-F1 v3.1 − tcn/wp100: Δ +0.019 [-0.01…+0.03], P = 0.77 → tidak terbedakan  (lebih baik = v3.1 lebih baik)


Bootstrap bersarang bigru/wp100: 0.830 [0.68–0.91] | vs v1 Δ +0.086 [+0.01…+0.16], P = 0.98 → lebih baik


Bootstrap bersarang tcn/wp100: 0.830 [0.67–0.90] | vs v1 Δ +0.086 [+0.01…+0.14], P = 0.98 → lebih baik


## 6. Simpan semua angka (dibaca `laporan/build_report.py --extract`)

In [8]:
keluaran = {
    "setelan": [f"{r['model']}/{r['nama_setelan']}" for r in semua], "acuan_metrik": acuan,
    "model_acuan": MODEL_ACUAN, "antar_model": antar_model, "balancing": hasil_balancing, "tuning": hasil_tuning,
    "kandidat": KANDIDAT, **hasil_kandidat, "lingkungan": evaluasi._lingkungan(),
}
p = data.lokasi_hasil() / "gabungan.json"
p.write_text(json.dumps(keluaran, indent=1, default=float) + "\n", encoding="utf-8")
print(f"{p} ({p.stat().st_size / 1024:.0f} KB)")

/mnt/HDD/03_Akademik_dan_Tugas/WrapSihedaf/cache/v11/gabungan.json (24 KB)


## Kesimpulan tim

Dijalankan 3 November 2026 di PC (CPU) lewat `tools/run_notebook.py`, 55 menit, 31 setelan dari keenam model. Laporan: `laporan/laporan_v11_model_tim.pdf`. Semua angka + bias kelas cross-fit, CI 95% bootstrap pasien.

- **Tidak ada model yang lebih baik dari v3.1** (0,832 [0,67–0,92]). Dengan resep yang sama (`bawaan`), BiGRU terbaik: 0,821 [0,68–0,87], tidak terbedakan dari v3.1 (P = 0,34). Vonis vs v3.1 model lain: SVM 0,773 (P = 0,072, tidak terbedakan); InceptionTime 0,767 (P = 0,006, lebih buruk); TCN 0,752 (P = 0,035, lebih buruk); XGBoost 0,730 (P = 0,004, lebih buruk); MiniRocket 0,709 (P = 0,000, lebih buruk).
- **Antar model, resep sama (vs BiGRU bawaan):** SVM −0,048 (P = 0,162); InceptionTime −0,055 (P = 0,022); TCN −0,070 (P = 0,003); XGBoost −0,091 (P = 0,000); MiniRocket −0,112 (P = 0,000). Arsitektur berpengaruh dengan bobot kelas lunak, terutama pada AFL 2:1 (pasien 08378).
- **Bobot kelas penuh (kandidat):** BiGRU wp100 0,830 [0,69–0,93], vs v3.1 −0,002 (P = 0,47), tanpa 04936 0,852 (P(v3.1 lebih baik) = 0,60), bersarang vs v1 P = 0,980; TCN wp100 0,830 [0,67–0,91], vs v3.1 −0,002 (P = 0,38), tanpa 04936 0,840 (P(v3.1 lebih baik) = 0,77), bersarang vs v1 P = 0,984. v3.1 tanpa 04936: 0,859. Setara v3.1, tidak lebih baik. InceptionTime wp100 tidak naik (0,766): keyakinan AFL di window non-AFL ikut naik.
- **Balancing (arahan pembimbing), vs bawaan model yang sama:** BiGRU over −0,045 (P = 0,000); BiGRU under −0,223 (P = 0,000); XGBoost smote +0,013 (P = 0,977); XGBoost under +0,063 (P = 0,984). Untuk BiGRU lebih buruk; untuk XGBoost lebih baik dari bawaan, tetapi XGBoost under tetap di bawah v3.1 (P = 0,001). Bawaan v11 tetap tanpa balancing.
- **Tuning cross-fit (perkiraan jujur model yang di-tuning):** BiGRU 0,818 (vs v3.1 P = 0,08), TCN 0,782 (vs v3.1 P = 0,25), InceptionTime 0,766 (vs v3.1 P = 0,01), XGBoost 0,745 (vs v3.1 P = 0,00).
- **Takikardia (butir 9):** ambang sintetis 115–140 bpm (v3.1 115), tetapi alarm pada window Normal asli ≥ 120 bpm 71,8–92,4% (v3.1 95,2%). Belum terpecahkan oleh model RR mana pun.
- **Per pasien:** AFL 2:1 pasien 08378 adalah pembeda utama (recall + bias: BiGRU 0,74, SVM 0,98, InceptionTime 0,79, TCN 0,01, XGBoost 0,05, MiniRocket 0,11; kandidat 1,00, 0,92). Pasien 07910 dan 08215 (AFL lambat/tidak teratur) hampir tidak terdeteksi oleh v3.1 maupun model v11 mana pun.
- **Yang tidak bisa disimpulkan:** setelan selain bawaan dipilih setelah melihat hasil (karena itu ada seleksi cross-fit); model dilatih di mesin berbeda; MiniRocket belum konvergen di 11 dari 25 fold; SVM, InceptionTime, dan MiniRocket hanya punya 1–2 setelan. **Paket produksi tetap v3.1.**